# MAVE data analysis II - From read counts to scores
In the previous session, you will have learned how to analyse a saturation genome editing (SGE) experiment starting with the raw sequencing reads in FASTQ format and counting the observations of each variant in the dataset. The output of this process is a count matrix or count table.

Now we will use a simple statistical method to turn the count matrix into scores, or variant effect measurements. The scoring method aims to capture the relative enrichment or depletion of each variant over the course of the experiment, so that we can use the scores for each variant for downstream inference and interpretation.

There is a very wide diversity of statistical methods for calculating scores for MAVE assays that attempt to capture or account for specific experimental design choices. Today we’ll use one of the oldest and most generic approaches for demonstration purposes. For a review of the many existing methods, see this paper:

Çubuk, H., Jin, X., Phipson, B., Marsh, J. A., & Rubin, A. F. (2025). Variant scoring tools for deep mutational scanning. Molecular systems biology, 21(10), 1293–1305. https://doi.org/10.1038/s44320-025-00137-x


## Example dataset
For this example, we'll be using the BAP1 SGE dataset that has already been processed and shared on the BioStudies portal: https://www.ebi.ac.uk/biostudies/studies/S-BSST1222

Download the `counts.tar.gz` file and extract the two files for exon 5, the same exon we analysed in the previous session. These are:
- `E5_SGA_count_frame.csv`
- `E5_SGB_count_frame.csv`

The two files provide the counts for each of two different guides used to target BAP1 exon 5.

## Loading count data
The first step of this tutorial is to load the count data into the notebook environment. We’ll use the `pandas` library for this:

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
count_df = pd.read_csv("E5_SGA_count_frame.csv")
count_df

The headings in the file include counts for each day when samples were collected for sequencing and each of three different replicates.

The `id` column lists the variant identifier, which is not in a standard format but instead contains multiple pieces of pertinent information about the variant concatenated together.

## Score calculation basics
The scoring function we are going to use today is described by the following equation:

$ s_v = \log_2 \left( \frac{ \left(\frac{c_{v, \text{output}}}{\sum_{i} c_{i, \text{output}}}\right)}{ \left(\frac{c_{v, \text{input}}}{\sum_{i} c_{i, \text{input}}}  \right)} \right) $

The score of a given variant $s_v$ is the log-transformed ratio of the variant’s frequency in the output library (our later time point) to the variant’s frequency in the input library (our earlier time point).

Using this approach, variants that are depleted over time will have a low score and variants whose frequency changes very little will have a score close to zero.

We can apply this scoring function to a specific replicate and pair of time points to calculate scores:

In [ ]:
scores = np.log2(count_df['D21R1'] / count_df['D21R1'].sum()) - np.log2(count_df['D4R1'] / count_df['D4R1'].sum())

In [ ]:
scores

We can add this vector as a new column in our counts data frame, or we can make a new data frame with just the counts we used and these new scores.

In [ ]:
score_df = pd.DataFrame(count_df[['id', 'D4R1', 'D21R1']])
score_df['score_D4_D21_R1'] = scores
score_df

Now let's calculate the scores for the same time points but with another replicate and compare the results.

In [ ]:
score_df = pd.concat([score_df, count_df[['D4R2', 'D21R2']]], axis=1)
score_df['score_D4_D21_R2']  = np.log2(score_df['D21R2'] / score_df['D21R2'].sum()) - np.log2(score_df['D4R2'] / score_df['D4R2'].sum())
score_df

You will have seen a divide by zero error. This is because one of the variants in this replicate had no counts for day 21, replicate 2.

When encountering something like this, it's always a good idea to inspect the data directly to confirm it:

In [ ]:
score_df.loc[score_df['score_D4_D21_R2'] == -np.inf]

There are multiple different ways to approach this.
One is to just leave the `-inf` in the data frame, but this can cause problems with aggregate calculations later.
For example, since negative infinity is one of the values, the mean of this column is negative infinity, which is not very useful.

Instead, sometimes researchers will add a pseudocount (either adding 1 or 0.5 so that there are no zeroes), or more commonly they will use count filtering to remove low-count variants from the dataset.
Count filtering is also popular because it removes low count variants whose characteristics are likely to be dominated by sampling noise.

## Count filtering
One way to apply count filtering using `pandas` is to set any count lower than a certain value to `pd.NA`, which will make the result of any calculation it's based on also `pd.NA`.
The library provides some extra options to skip these values, so we can still get the sum of counts.

Let's calculate the scores for day 4 and day 21 for all three replicates in a new data frame, but apply a count filter of at least 100 reads.

In [ ]:
score_df_v2 = pd.DataFrame(count_df[['id', 'D4R1', 'D21R1', 'D4R2', 'D21R2', 'D4R3', 'D21R3']])
score_df_v2[score_df_v2.loc[:, score_df_v2.columns != 'id'] < 100] = pd.NA
score_df_v2[score_df_v2.isna().any(axis=1)]

Now let's calculate the scores for the three replicates.

In [ ]:
d1 = 4
d2 = 21
for rep in range(1,4):
    score_df_v2[f'score_D{d1}_D{d2}_R{rep}']  = np.log2(score_df_v2[f'D{d2}R{rep}'] / score_df_v2[f'D{d2}R{rep}'].sum()) - np.log2(score_df_v2[f'D{d1}R{rep}'] / score_df_v2[f'D{d1}R{rep}'].sum())
score_df_v2

In [ ]:
score_df_v2[score_df_v2.isna().any(axis=1)]

We can see that any variants with `NaN` values were not scored, which is what we were hoping to see.

## Comparisons and exploration
Now that we have some scores, we can think about how to explore and interpret the data.
One standard quality control exercise we can do is to calculate the correlations of the scores across replicates.
We can also visualise the replicate correlations.

In [ ]:
score_df_v2.filter(like="score").corr()

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(score_df_v2['score_D4_D21_R1'], score_df_v2['score_D4_D21_R2'], alpha=0.6)

## Extension
Now you can try working with the data yourself. Here are some ideas:
- Compare different pairs of time points and see how the scores change
- Try other filtering parameters
- Load the data from the other guide and compare across those two datasets
- Download the real scores from the BioStudies portal and see how your simple scores compare